### **Model Training**

### Import Data and Required Packages

In [21]:
import numpy as np
import pandas as pd
import pickle
import warnings
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
import seaborn as sns

# Sklearn
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer

from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier,AdaBoostClassifier
from sklearn.svm import SVC

# Imbalanced data
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

### Import the CSV Data as Pandas DataFrame

In [22]:
data = pd.read_csv("G:\\check\\Social_media_impact_on_life.csv")

In [23]:
data.shape

(1705, 11)

### Show Top 5 Records

In [24]:
data.head()

,Student_ID,Age,Gender,Academic_Level,Country,Avg_Daily_Usage_Hours,Most_Used_Platform,Affects_Academic_Performance,Sleep_Hours_Per_Night,Mental_Health_Score,Overall_Impact
0,232,21,Male,Undergraduate,Other,4.0,Facebook,No,6.7,6.8,Neutral
1,564,23,Female,Undergraduate,Other,1.6,LinkedIn,No,8.6,7.6,Positive
2,788,22,Male,Graduate,Canada,4.6,Instagram,No,6.7,7.0,Neutral
3,686,18,Male,Undergraduate,Other,7.0,Snapchat,Yes,5.4,5.3,Negative
4,608,24,Female,High School,Other,7.5,Facebook,Yes,5.0,4.4,Negative


### Preparing X and Y variables

In [25]:
X = data.drop(columns=['Overall_Impact', 'Student_ID'])

In [26]:
X

,Age,Gender,Academic_Level,Country,Avg_Daily_Usage_Hours,Most_Used_Platform,Affects_Academic_Performance,Sleep_Hours_Per_Night,Mental_Health_Score
0,21,Male,Undergraduate,Other,4.0,Facebook,No,6.7,6.8
1,23,Female,Undergraduate,Other,1.6,LinkedIn,No,8.6,7.6
2,22,Male,Graduate,Canada,4.6,Instagram,No,6.7,7.0
3,18,Male,Undergraduate,Other,7.0,Snapchat,Yes,5.4,5.3
4,24,Female,High School,Other,7.5,Facebook,Yes,5.0,4.4
...,...,...,...,...,...,...,...,...,...
1700,20,Female,Undergraduate,Italy,4.7,TikTok,No,7.2,7.0
1701,23,Male,Graduate,Russia,6.8,Instagram,Yes,5.9,4.0
1702,21,Female,Undergraduate,China,5.6,WeChat,Yes,6.7,6.0
1703,24,Male,Graduate,Japan,4.3,Twitter,No,7.5,8.0


In [27]:
y=data['Overall_Impact']

In [28]:
y

0        Neutral
1       Positive
2        Neutral
3       Negative
4       Negative
          ...   
1700    Positive
1701    Negative
1702    Negative
1703    Positive
1704    Negative
Name: Overall_Impact, Length: 1705, dtype: object

In [29]:
print("Categories in 'Gender' variable:     ",end=" " )
print(data['Gender'].unique())

print("Categories in 'Academic_Level' variable:  ",end=" ")
print(data['Academic_Level'].unique())

print("Categories in'Country' variable:",end=" " )
print(data['Country'].unique())

print("Categories in 'Most_Used_Platform' variable:     ",end=" " )
print(data['Most_Used_Platform'].unique())

print("Categories in 'Affects_Academic_Performance' variable:     ",end=" " )
print(data['Affects_Academic_Performance'].unique())

print("Categories in 'Overall_Impact' variable:     ",end=" " )
print(data['Overall_Impact'].unique())

Categories in 'Gender' variable:      ['Male' 'Female']
Categories in 'Academic_Level' variable:   ['Undergraduate' 'Graduate' 'High School']
Categories in'Country' variable: ['Other' 'Canada' 'USA' 'India' 'Australia' 'UK' 'Germany' 'Bangladesh'
 'Brazil' 'Japan' 'South Korea' 'France' 'Spain' 'Italy' 'Mexico' 'Russia'
 'China' 'Sweden' 'Norway' 'Denmark' 'Netherlands' 'Belgium' 'Switzerland'
 'Austria' 'Portugal' 'Greece' 'Ireland' 'New Zealand' 'Singapore'
 'Malaysia' 'Thailand' 'Vietnam' 'Philippines' 'Indonesia' 'Taiwan'
 'Hong Kong' 'Turkey' 'Israel' 'UAE' 'Egypt' 'Morocco' 'South Africa'
 'Nigeria' 'Kenya' 'Ghana' 'Argentina' 'Chile' 'Colombia' 'Peru'
 'Venezuela' 'Ecuador' 'Uruguay' 'Paraguay' 'Bolivia' 'Costa Rica'
 'Panama' 'Jamaica' 'Trinidad' 'Bahamas' 'Iceland' 'Finland' 'Poland'
 'Romania' 'Hungary' 'Czech Republic' 'Slovakia' 'Croatia' 'Serbia'
 'Slovenia' 'Bulgaria' 'Estonia' 'Latvia' 'Lithuania' 'Ukraine' 'Moldova'
 'Belarus' 'Kazakhstan' 'Uzbekistan' 'Kyrgyzstan' 'Taj

In [30]:
## define numerical and categorical features
numerical_features = X.select_dtypes(include=['int64','float64']).columns

categorical_features = X.select_dtypes(include=['object']).columns
categorical_features = categorical_features.drop('Academic_Level')

orinal_categorical_featres = ['Academic_Level']

print(numerical_features)
print(categorical_features)
print(orinal_categorical_featres)

Index(['Age', 'Avg_Daily_Usage_Hours', 'Sleep_Hours_Per_Night',
       'Mental_Health_Score'],
      dtype='object')
Index(['Gender', 'Country', 'Most_Used_Platform',
       'Affects_Academic_Performance'],
      dtype='object')
['Academic_Level']


In [31]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder,StandardScaler,OrdinalEncoder

numerical_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(handle_unknown='ignore')
ordinal_categorical_transformer =  OrdinalEncoder(
    categories=[['High School', 'Undergraduate', 'Graduate']]
)
preprocessor = ColumnTransformer(
    transformers=[
        ("OneHotEncoder",categorical_transformer,categorical_features),
        ("OrdinalEncoder",ordinal_categorical_transformer,orinal_categorical_featres),
        ("StandardScaler",numerical_transformer,numerical_features)
    ]
)

In [32]:
# separate dataset into train and test
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)
X_train.shape, X_test.shape

((1364, 9), (341, 9))

### **Checking Target Variable is Balanced or Imbalanced**

In [33]:
data['Overall_Impact'].value_counts(normalize=True)

Overall_Impact
Negative    0.550733
Positive    0.292669
Neutral     0.156598
Name: proportion, dtype: float64

### Insight
- So the target is imbalanced we are applying SMOTE

### **Models**

In [34]:
models = {
    "LogisticRegression": {
        "model": LogisticRegression(max_iter=1000),
        "params": {
            "model__C": [0.01, 0.1, 1, 10]
        }
    },

    "DecisionTree": {
        "model": DecisionTreeClassifier(),
        "params": {
            "model__max_depth": [3, 5, 10, None],
            "model__min_samples_split": [2, 5, 10]
        }
    },

    "RandomForest": {
        "model": RandomForestClassifier(),
        "params": {
            "model__n_estimators": [100, 200],
            "model__max_depth": [5, 10, None]
        }
    },

    "SVC": {
        "model": SVC(),
        "params": {
            "model__C": [0.1, 1, 10],
            "model__kernel": ['linear', 'rbf']
        }
    },

    "KNN":{
        "model":KNeighborsClassifier(),
        "params":{
            "model__n_neighbors":[5,7,9,11,15],
            "model__weights":["distance","uniform"]
        }
    },

    "AdaBoostClassifer":{
        "model":AdaBoostClassifier(),
        "params":{
            "model__learning_rate":[1.0,3.0,5.0]
        }
    }
}

### **Training Tuneing**

In [35]:
best_models = {}
results = []

for name, config in models.items():

    print(f"\n🔹 Training {name}...")

    # Pipeline: Preprocessing + SMOTE + Model
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("smote", SMOTE(random_state=42)),
        ("model", config["model"])
    ])

    grid = GridSearchCV(
        pipeline,
        config["params"],
        cv=5,
        scoring='accuracy',
        n_jobs=-1
    )

    grid.fit(X_train, y_train)

    best_model = grid.best_estimator_

    # Predictions
    y_pred = best_model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')

    print("Best Params:", grid.best_params_)
    print("Accuracy:", acc)
    print("F1 Score:", f1)

    best_models[name] = best_model
    results.append((name, acc))


🔹 Training LogisticRegression...
Best Params: {'model__C': 1}
Accuracy: 0.9384164222873901
F1 Score: 0.939668286323311

🔹 Training DecisionTree...
Best Params: {'model__max_depth': None, 'model__min_samples_split': 5}
Accuracy: 0.9912023460410557
F1 Score: 0.9911822806882016

🔹 Training RandomForest...
Best Params: {'model__max_depth': None, 'model__n_estimators': 100}
Accuracy: 0.9882697947214076
F1 Score: 0.988169737440293

🔹 Training SVC...
Best Params: {'model__C': 10, 'model__kernel': 'rbf'}
Accuracy: 0.9824046920821115
F1 Score: 0.9823015935686572

🔹 Training KNN...
Best Params: {'model__n_neighbors': 5, 'model__weights': 'distance'}
Accuracy: 0.9589442815249267
F1 Score: 0.9600200571077563

🔹 Training AdaBoostClassifer...
Best Params: {'model__learning_rate': 5.0}
Accuracy: 0.8152492668621701
F1 Score: 0.8347608415653478


### **Best Model**

In [36]:
results_df = pd.DataFrame(results, columns=['Model', 'Accuracy'])
results_df = results_df.sort_values(by='Accuracy', ascending=False)

print("\n Model Comparison:")
print(results_df)

best_model_name = results_df.iloc[0]['Model']
best_model = best_models[best_model_name]

print(f"\n Best Model: {best_model_name}")


y_pred = best_model.predict(X_test)

print("\n Classification Report:")
print(classification_report(y_test, y_pred))


 Model Comparison:
                Model  Accuracy
1        DecisionTree  0.991202
2        RandomForest  0.988270
3                 SVC  0.982405
4                 KNN  0.958944
0  LogisticRegression  0.938416
5   AdaBoostClassifer  0.815249

 Best Model: DecisionTree

 Classification Report:
              precision    recall  f1-score   support

    Negative       0.99      1.00      1.00       197
     Neutral       0.98      0.98      0.98        51
    Positive       0.99      0.98      0.98        93

    accuracy                           0.99       341
   macro avg       0.99      0.99      0.99       341
weighted avg       0.99      0.99      0.99       341



### **Saveing Model**

In [37]:
with open("best_model.pkl", "wb") as f:
    pickle.dump(best_model, f)

print("\n💾 Model saved successfully!")


💾 Model saved successfully!


In [38]:
import sklearn
sklearn.__version__

'1.6.1'

In [39]:
import joblib
joblib.__version__

'1.4.2'